# Gráficas de línea por variable

Visualización de cada variable del dataset `synthetic_monitoring_activity.csv`.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("synthetic_monitoring_activity.csv", parse_dates=["timestamp_local"])
print(f"Registros: {len(df)}")
print(f"Rango: {df['timestamp_local'].min()} → {df['timestamp_local'].max()}")
print(f"Columnas numéricas: {list(df.select_dtypes(include='number').columns)}")

## 1. Series temporales completas (3 meses)

In [ ]:
cols = ["actividad", "http_4xx", "http_5xx", "sum_downloadTime", "avg_downloadTime"]
colors = ["#636EFA", "#EF553B", "#AB63FA", "#00CC96", "#FFA15A"]

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=cols)

for i, (col, color) in enumerate(zip(cols, colors), 1):
    fig.add_trace(go.Scattergl(x=df["timestamp_local"], y=df[col],
                               mode="lines", name=col,
                               line=dict(width=0.5, color=color)), row=i, col=1)

fig.update_layout(height=1200, showlegend=False, title_text="Series temporales completas")
fig.show()

## 2. Zoom: una semana laboral (6-12 enero)

In [ ]:
week = df[(df["timestamp_local"] >= "2025-01-06") & (df["timestamp_local"] < "2025-01-13")]

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=cols)

for i, (col, color) in enumerate(zip(cols, colors), 1):
    fig.add_trace(go.Scatter(x=week["timestamp_local"], y=week[col],
                             mode="lines", name=col,
                             line=dict(width=1, color=color)), row=i, col=1)

fig.update_layout(height=1200, showlegend=False, title_text="Semana 6-12 enero 2025")
fig.show()

## 3. Zoom: un día completo (7 enero, martes)

In [ ]:
day = df[df["eventdate"] == "2025-01-07"]

fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=cols)

for i, (col, color) in enumerate(zip(cols, colors), 1):
    fig.add_trace(go.Scatter(x=day["timestamp_local"], y=day[col],
                             mode="lines+markers", name=col,
                             line=dict(width=1.5, color=color),
                             marker=dict(size=3)), row=i, col=1)

fig.update_layout(height=1200, showlegend=False, title_text="Martes 7 enero 2025")
fig.show()

## 4. Comparación: día laborable vs fin de semana

In [ ]:
lab = df[df["eventdate"] == "2025-01-07"]  # martes
fds = df[df["eventdate"] == "2025-01-11"]  # sábado

fig = make_subplots(rows=5, cols=1, shared_xaxes=False, vertical_spacing=0.03,
                    subplot_titles=[f"{c} (azul=laborable, rojo=fin de semana)" for c in cols])

for i, col in enumerate(cols, 1):
    fig.add_trace(go.Scatter(x=lab["timestamp_local"].dt.hour + lab["timestamp_local"].dt.minute/60,
                             y=lab[col], mode="lines", name=f"{col} lab",
                             line=dict(color="#636EFA"), showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=fds["timestamp_local"].dt.hour + fds["timestamp_local"].dt.minute/60,
                             y=fds[col], mode="lines", name=f"{col} fds",
                             line=dict(color="#EF553B"), showlegend=(i==1)), row=i, col=1)

fig.update_layout(height=1200, title_text="Laborable (mar 7 ene) vs Fin de semana (sáb 11 ene)")
fig.show()